# Commodity forecasting | Data and prediction contract

A research project built around 424 multi-horizon return targets from LME, JPX, US equities, and FX. This notebook establishes what can be known at prediction time and which data is reserved for final evaluation.

**Research question:** can economically motivated representations improve stable cross-sectional return ranking?

[Dataset](https://www.kaggle.com/competitions/mitsui-commodity-prediction-challenge/data) · [Official target construction](https://www.kaggle.com/code/sohier/mitsui-target-calculation-example/) · Next: `01_eda.ipynb`.

In [ ]:
from pathlib import Path
import sys
if not Path("pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent))
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from scripts.notebook_support import project_root, checked_reports, show_figure
root = project_root()
audit, research = checked_reports(root)
config = json.loads((root / "configs/research.json").read_text())
display(Markdown(f"**Verified experiment:** `{research['lineage'][:16]}` · **Feature gate:** open"))

In [ ]:
from commodity_prediction.data import load_data, make_folds
x, y, pairs = load_data(root)
folds, development_stop = make_folds(len(x), config)
inventory = pd.DataFrame({"Measure": ["Observed dates", "Input columns", "Return targets", "Development dates", "Reserved holdout dates"],
                          "Count": [len(x), x.shape[1], y.shape[1], development_stop, len(x) - development_stop]})
display(inventory.set_index("Measure"))

## The target starts after the current date

For an asset with price $P$ and horizon $h$, the target at date $t$ is $\log(P_{t+h+1}/P_{t+1})$. For a pair, subtract the second asset's return. Current-row market observations are available before prediction. Labels become available at $t+h+1$.

This is why a one-day target needs a two-date availability delay. Shifting targets by only the stated horizon would leak information.

In [ ]:
release = pd.DataFrame({"Horizon": [1, 2, 3, 4], "Label release delay": [2, 3, 4, 5], "Targets": [int((pairs.lag == h).sum()) for h in range(1, 5)]})
display(release.set_index("Horizon"))
display(Markdown(f"Target reconstruction compared **{audit['target_reconstruction_compared_values']:,}** observable development labels. Maximum absolute difference: **{audit['target_reconstruction_max_absolute_error']:.2e}**, below the 1e-5 rounding tolerance."))

## Reserve the final 252 dates

Three expanding walk-forward folds use 180 validation dates each. Five dates are purged before every validation block, so all fitting labels were released strictly before its first prediction. The final 252 dates are excluded from feature construction, EDA, screening, and model selection. The downloadable mock test file overlaps training and is not a valid holdout.

In [ ]:
fig = go.Figure()
for fold in folds:
    label = f"Fold {fold.number + 1}"
    for part, start, length, color in [
        ("Train", 0, fold.train_stop, "#1F6C99"),
        ("Purge", fold.train_stop, fold.validation_start - fold.train_stop, "#EDAF43"),
        ("Validation", fold.validation_start, fold.validation_stop - fold.validation_start, "#27A394"),
        ("Reserved holdout", development_stop, len(x) - development_stop, "#CAD3DE")]:
        fig.add_trace(go.Bar(x=[length], y=[label], base=start, orientation="h", name=part,
                             marker_color=color, showlegend=fold.number == 0))
fig.update_layout(barmode="overlay", title="Validation respects prediction-time information", xaxis_title="Date index", legend={"orientation": "h", "y": -0.22})
show_figure(fig, root, "validation_protocol", 450)

## Reproducibility and data handling

The public repository contains source, aggregate evidence, and executed notebooks. Raw market rows, labels, and model predictions remain in private storage under the competition's data-use terms. A data/source/configuration fingerprint connects every downstream result. Missing, changed, or corrupt checkpoints cause a clear failure rather than silent reuse.

**Status:** data contract verified. Feature research is in progress; final evaluation has not run.